In [1]:
#from data_quality_test_management import *
#from recipe_management import *
from dotenv import load_dotenv
import os
# Charger les variables du fichier .env
load_dotenv()
from rapidfuzz import fuzz, process
import re
from data_quality_test_management import *

In [ ]:
recipes_df = pd.read_csv("recipes.csv")
recipe_instructions_df = pd.read_csv("recipe_instructions.csv")
recipe_ingredients_df = pd.read_csv("recipe_ingredients.csv")
graph_brut  = pd.read_csv("data/dataset_graph_recette.csv")
graph_nettoye =  pd.read_csv("data/final_dataset_cleaned_with_non_gestures.csv")
graph_geste_only  = pd.read_csv("data/final_dataset_cleaned_without_non_gestures.csv")


In [2]:
if __name__ == "__main__":
   
    
    # Chemins des fichiers
    RECIPES_CSV = "recipes.csv"
    GRAPHS_RECIPES_CSV = "data/dataset_graph_recette.csv"
    GRAPHS_CLEANED_WITH_GESTURES_CSV = "data/final_dataset_cleaned_with_non_gestures.csv"
    GRAPHS_CLEANED_WITHOUT_GESTURES_CSV = "data/final_dataset_cleaned_without_non_gestures.csv"
    
    # Exécuter le pipeline
    run_strategy_2_pipeline(
        recipes_csv=RECIPES_CSV,
        graphs_recipes_csv=GRAPHS_RECIPES_CSV,
        graphs_cleaned_with_gestures_csv=GRAPHS_CLEANED_WITH_GESTURES_CSV,
        graphs_cleaned_without_gestures_csv=GRAPHS_CLEANED_WITHOUT_GESTURES_CSV,
        output_dir="strategy_2_results"
    )


################################################################################
# PIPELINE STRATÉGIE 2 - VALIDATION STRUCTURELLE (VERSION AMÉLIORÉE)
################################################################################

📂 CHARGEMENT DES DONNÉES...
  ✅ Recettes: 1,029,720
  ✅ Graphes bruts: 4,073,652
  ✅ Graphes nettoyés (avec): 2,749,491
  ✅ Graphes nettoyés (sans): 2,241,617

🏷️  CLASSIFICATION DES RECETTES...
  ✅ Colonne 'category' ajoutée
     Distribution : {'other': 475684, 'bakery': 285774, 'quick_prep': 165490, 'stew': 102772}
  ✅ Colonne 'complexity' ajoutée
     Distribution : {'simple': 624238, 'moyenne': 327087, 'elevee': 78395}
  ✅ Recipes enrichi sauvegardé : strategy_2_results\recipes_enriched.csv

EXÉCUTION DU TEST 1 SUR LES 3 DATASETS

🔷 TEST 1 - DATASET BRUT (AVANT NETTOYAGE)

TEST 1 : CALCUL DE LA TAILLE DES LISTES D'ACTIONS

📊 STATISTIQUES GLOBALES
--------------------------------------------------------------------------------
Total graphes: 4,073,652
L

In [38]:
data_synthese  =  pd.read_csv("strategy_2_results/dataset_synthese.csv")


In [39]:


# Filtrer les recettes avec flags dans le Test 2
test2_flags = data_synthese[data_synthese['test'] == '2']

# Extraire les IDs uniques
ids_test2_flags = test2_flags['id'].unique()

print(f"Nombre de recettes avec flags Test 2 : {len(ids_test2_flags):,}")

# Sauvegarder dans un fichier
pd.DataFrame({'id': ids_test2_flags}).to_csv("recettes_test2_flags.csv", index=False)

# Optionnel : séparer par type de flag
flags_ratio_faible = test2_flags[test2_flags['flag'] == 'FLAG_RATIO_FAIBLE']['id'].unique()
flags_ratio_eleve = test2_flags[test2_flags['flag'] == 'FLAG_RATIO_ELEVE']['id'].unique()

print(f"  - FLAG_RATIO_FAIBLE : {len(flags_ratio_faible):,} recettes")
print(f"  - FLAG_RATIO_ELEVE : {len(flags_ratio_eleve):,} recettes")

Nombre de recettes avec flags Test 2 : 173,336
  - FLAG_RATIO_FAIBLE : 166,798 recettes
  - FLAG_RATIO_ELEVE : 6,538 recettes


In [42]:
import pandas as pd

# Charger le dataset synthèse
df = data_synthese.copy()

# Extraire les IDs uniques par test
ids_test2 = set(df[df['test'] == '2']['id'].unique())
ids_test3 = set(df[df['test'] == '3']['id'].unique())
ids_test4a = set(df[df['test'] == '4A']['id'].unique())
ids_test4b = set(df[df['test'] == '4B']['id'].unique())
ids_test5 = set(df[df['test'] == '5']['id'].unique())
ids_test6 = set(df[df['test'] == '6']['id'].unique())

print(f"IDs flaggés par test :")
print(f"  Test 2 : {len(ids_test2):,}")
print(f"  Test 3 : {len(ids_test3):,}")
print(f"  Test 4A : {len(ids_test4a):,}")
print(f"  Test 4B : {len(ids_test4b):,}")
print(f"  Test 5 : {len(ids_test5):,}")
print(f"  Test 6 : {len(ids_test6):,}")

# Comparaisons utiles
print(f"\n--- Comparaisons ---")
print(f"Test2 ∩ Test3 : {len(ids_test2 & ids_test3):,}")
print(f"Test2 ∩ Test5 : {len(ids_test2 & ids_test5):,}")
print(f"Test3 ∩ Test5 : {len(ids_test3 & ids_test5):,}")
print(f"Test2 ∩ Test3 ∩ Test5 : {len(ids_test2 & ids_test3 & ids_test5):,}")

# IDs uniquement dans Test2 (pas dans les autres tests critiques)
ids_only_test2 = ids_test2 - ids_test3 - ids_test5 - ids_test6
print(f"\nIDs uniquement dans Test2 : {len(ids_only_test2):,}")

# Union de tous les IDs flaggés
all_flagged_ids = ids_test2 | ids_test3 | ids_test4a | ids_test4b | ids_test5 | ids_test6
print(f"Total IDs uniques flaggés : {len(all_flagged_ids):,}")

IDs flaggés par test :
  Test 2 : 173,336
  Test 3 : 80,542
  Test 4A : 566,625
  Test 4B : 143,659
  Test 5 : 31,955
  Test 6 : 191

--- Comparaisons ---
Test2 ∩ Test3 : 13,559
Test2 ∩ Test5 : 5,012
Test3 ∩ Test5 : 3,662
Test2 ∩ Test3 ∩ Test5 : 612

IDs uniquement dans Test2 : 155,377
Total IDs uniques flaggés : 710,813


In [2]:
"""
PIPELINE OPTIMISÉ DE NETTOYAGE DES VERBES CULINAIRES
=====================================================
Combine normalisation + filtrage des non-gestes avec RapidFuzz
Temps estimé : quelques secondes pour des millions de lignes

Auteur: Assistant Claude pour projet LIARA/UQAC
"""

import re
import ast
import time
from pathlib import Path
from functools import lru_cache
from typing import List, Dict, Set, Union, Optional, Tuple
from dataclasses import dataclass, field

# Import conditionnel de pandas (optionnel)
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

# RapidFuzz pour le matching flou ultra-rapide
from rapidfuzz import fuzz, process
from rapidfuzz.distance import Levenshtein


# =============================================================================
# DICTIONNAIRE DE NORMALISATION COMPLET
# =============================================================================

MAPPING_VERBES = {
    # =========================================================================
    # 1. VARIANTES ORTHOGRAPHIQUES (UK/US, accents, typos)
    # =========================================================================
    'mould': 'mold',
    'colour': 'color',
    'flavour': 'flavor',
    'pulverise': 'pulverize',
    'marbleise': 'marbleize',
    'texturise': 'texturize',
    'tenderise': 'tenderize',
    'caramelise': 'caramelize',
    'crystallise': 'crystallize',
    'spiralise': 'spiralize',
    'liquidise': 'liquidize',
    'brûlée': 'brulee',
    'flambé': 'flambe',
    'sauté': 'saute',
    'purée': 'puree',
    'julienned': 'julienne',
    'wisk': 'whisk',
    'whizz': 'whiz',
    'garish': 'garnish',
    'rins': 'rinse',
    
    # =========================================================================
    # 2. SYNONYMES D'ÉCRASEMENT
    # =========================================================================
    'smash': 'mash',
    'smoosh': 'mash',
    'moosh': 'mash',
    'smush': 'mash',
    'mush': 'mash',
    
    # =========================================================================
    # 3. SYNONYMES DE MÉLANGE
    # =========================================================================
    'amalgamate': 'combine',
    'admixture': 'mix',
    'meld': 'blend',
    'mingle': 'mix',
    'incorporate': 'mix',
    'reincorporate': 'mix',
    'recombine': 'combine',
    'reblend': 'blend',
    'premix': 'mix',
    'remash': 'mash',
    
    # =========================================================================
    # 4. SYNONYMES DE DÉCOUPE
    # =========================================================================
    'hacken': 'chop',
    'crosscut': 'cut',
    'trisect': 'cut',
    'cleave': 'cut',
    'sever': 'cut',
    'prechop': 'chop',
    
    # =========================================================================
    # 5. SYNONYMES DE PLACEMENT
    # =========================================================================
    'plonk': 'place',
    'plop': 'place',
    'plunk': 'place',
    'position': 'place',
    'reposition': 'place',
    'situate': 'place',
    
    # =========================================================================
    # 6. SYNONYMES DE VERSEMENT
    # =========================================================================
    'dribble': 'drizzle',
    'trickle': 'drizzle',
    'stream': 'pour',
    'decant': 'pour',
    
    # =========================================================================
    # 7. SYNONYMES DE SAUPOUDRAGE
    # =========================================================================
    'scatter': 'sprinkle',
    'strew': 'sprinkle',
    'shower': 'sprinkle',
    'dot': 'sprinkle',
    'fleck': 'sprinkle',
    
    # =========================================================================
    # 8. SYNONYMES D'ÉTALEMENT
    # =========================================================================
    'smear': 'spread',
    'slather': 'spread',
    'schmear': 'spread',
    
    # =========================================================================
    # 9. SYNONYMES DE NETTOYAGE
    # =========================================================================
    'cleanse': 'clean',
    'scrub': 'clean',
    'scour': 'clean',
    
    # =========================================================================
    # 10. SYNONYMES D'ENROBAGE
    # =========================================================================
    'enrobe': 'coat',
    'encrust': 'coat',
    'recoat': 'coat',
    'dredge': 'coat',
    
    # =========================================================================
    # 11. SYNONYMES DE RETRAIT
    # =========================================================================
    'discard': 'remove',
    'dispose': 'remove',
    'eliminate': 'remove',
    'extract': 'remove',
    
    # =========================================================================
    # 12. PRÉPARATION DES INGRÉDIENTS
    # =========================================================================
    'debone': 'bone',
    'deseed': 'seed',
    'de-seed': 'seed',
    'pare': 'peel',
    'skin': 'peel',
    'shuck': 'shell',
    'unshell': 'shell',
    
    # =========================================================================
    # 13. FILTRAGE/SÉPARATION
    # =========================================================================
    'sieve': 'sift',
    'strain': 'drain',
    
    # =========================================================================
    # 14. FORME/MODELAGE
    # =========================================================================
    'reshape': 'shape',
    'reform': 'form',
    'remold': 'mold',
    
    # =========================================================================
    # 15. OUVERTURE/FERMETURE
    # =========================================================================
    'uncover': 'open',
    'unwrap': 'open',
    'unfurl': 'open',
    'unfoil': 'open',
    'unseal': 'open',
    'reseal': 'seal',
    'close': 'seal',
    
    # =========================================================================
    # 16. RETOURNEMENT/ROTATION
    # =========================================================================
    'invert': 'turn',
    'rotate': 'turn',
    'revolve': 'turn',
    
    # =========================================================================
    # 17. PLIAGE/ENROULEMENT
    # =========================================================================
    'enfold': 'fold',
    'crease': 'fold',
    'coil': 'roll',
    'curl': 'roll',
    'spiral': 'roll',
    'twirl': 'roll',
    
    # =========================================================================
    # 18. PERFORATION
    # =========================================================================
    'poke': 'pierce',
    'prick': 'pierce',
    'puncture': 'pierce',
    'perforate': 'pierce',
    'stab': 'pierce',
    
    # =========================================================================
    # 19. TAPOTEMENT
    # =========================================================================
    'dab': 'pat',
    'tap': 'pat',
    'blot': 'pat',
    
    # =========================================================================
    # 20. ASSEMBLAGE
    # =========================================================================
    'build': 'assemble',
    'construct': 'assemble',
    'reassemble': 'assemble',
    
    # =========================================================================
    # 21. ATTACHEMENT
    # =========================================================================
    'fasten': 'tie',
    'secure': 'tie',
    'truss': 'tie',
    'bind': 'tie',
    'attach': 'tie',
    
    # =========================================================================
    # 22. GARNITURE/DÉCORATION
    # =========================================================================
    'adorn': 'garnish',
    'ornament': 'garnish',
    'decorate': 'garnish',
    
    # =========================================================================
    # 23. PRÉFIXES RE-
    # =========================================================================
    'readjust': 'adjust',
    'rearrange': 'arrange',
    'rebottle': 'bottle',
    'redraw': 'draw',
    'refill': 'fill',
    'reglaze': 'glaze',
    'rehydrate': 'hydrate',
    'reprocess': 'process',
    'reseason': 'season',
    'rewrap': 'wrap',
    'rework': 'work',
    'rewhisk': 'whisk',
    
    # =========================================================================
    # 24. PRÉFIXES PRE-
    # =========================================================================
    'premeasure': 'measure',
    'preseason': 'season',
    'presoak': 'soak',
    'prewarm': 'warm',
    
    # =========================================================================
    # 25. PRÉFIXES UN-
    # =========================================================================
    'uncurl': 'straighten',
    'untwist': 'straighten',
    'unwind': 'straighten',
    'unroll': 'open',
    'unmold': 'remove',
    'unskewer': 'remove',
    'unspit': 'remove',
    'unstack': 'separate',
    'unstuff': 'remove',
    'unthread': 'remove',
    'untie': 'open',
    'untruss': 'open',
    
    # =========================================================================
    # 26. VERBES COMPOSÉS
    # =========================================================================
    'tip out': 'pour',
    'wedge slice': 'slice',
    'egg wash': 'brush',
    'vacuum seal': 'seal',
    
    # =========================================================================
    # 27. FORMES CONJUGUÉES
    # =========================================================================
    'tipping': 'tip',
    'wrapper': 'wrap',
    'pieces': 'piece',
    'cooks': 'cook',
    'mixing': 'mix',
    'mixed': 'mix',
    'chopping': 'chop',
    'chopped': 'chop',
    'slicing': 'slice',
    'sliced': 'slice',
    'dicing': 'dice',
    'diced': 'dice',
    'stirring': 'stir',
    'stirred': 'stir',
    'beating': 'beat',
    'beaten': 'beat',
    'whisking': 'whisk',
    'whisked': 'whisk',
    'pouring': 'pour',
    'poured': 'pour',
    'folding': 'fold',
    'folded': 'fold',
    'kneading': 'knead',
    'kneaded': 'knead',
    'grating': 'grate',
    'grated': 'grate',
    'peeling': 'peel',
    'peeled': 'peel',
    'mincing': 'mince',
    'minced': 'mince',
    'crushing': 'crush',
    'crushed': 'crush',
    'mashing': 'mash',
    'mashed': 'mash',
    'spreading': 'spread',
    'spreaded': 'spread',
    'rolling': 'roll',
    'rolled': 'roll',
    'wrapping': 'wrap',
    'wrapped': 'wrap',
    'coating': 'coat',
    'coated': 'coat',
    'seasoning': 'season',
    'seasoned': 'season',
    'garnishing': 'garnish',
    'garnished': 'garnish',
    'draining': 'drain',
    'drained': 'drain',
    'straining': 'strain',
    'strained': 'strain',
    'sifting': 'sift',
    'sifted': 'sift',
    'crumbling': 'crumble',
    'crumbled': 'crumble',
    'flaking': 'flake',
    'flaked': 'flake',
    'shelling': 'shell',
    'shelled': 'shell',
    'piping': 'pipe',
    'piped': 'pipe',
    'tossing': 'toss',
    'tossed': 'toss',
    'shaking': 'shake',
    'shaked': 'shake',
    'sandwiching': 'sandwich',
    'sandwiched': 'sandwich',
    'processing': 'process',
    'processed': 'process',
    
    # =========================================================================
    # 28. ÉCLABOUSSURES
    # =========================================================================
    'splat': 'splash',
    'splatter': 'splash',
    'splodge': 'splash',
    'sploosh': 'splash',
    'splotch': 'splash',
    'spritz': 'spray',
    'squirt': 'spray',
    
    # =========================================================================
    # 29. PRESSER
    # =========================================================================
    'squish': 'squeeze',
    'squash': 'squeeze',
    
    # =========================================================================
    # 30. TECHNIQUES RARES
    # =========================================================================
    'chiffonade': 'slice',
    'concasse': 'chop',
    'tournee': 'cut',
    'quenelle': 'shape',
    'spatchcock': 'butterfly',
    'supreme': 'segment',
    
    # =========================================================================
    # 31. TREMPAGE/IMMERSION
    # =========================================================================
    'submerge': 'soak',
    'immerse': 'soak',
    'dunk': 'dip',
    'douse': 'soak',
    'drench': 'soak',
    'drown': 'soak',
    
    # =========================================================================
    # 32. VARIANTES REDONDANTES
    # =========================================================================
    'baggie': 'bag',
    'dollop': 'spoon',
    'ladle': 'spoon',
}


# =============================================================================
# VERBES NON-GESTUELS (à filtrer)
# =============================================================================

NON_GESTURE_VERBS = {
    # Cuissons passives
    'bake', 'roast', 'boil', 'simmer', 'steam', 'broil', 'grill', 'fry',
    'deep fry', 'shallow fry', 'pan fry', 'stir fry', 'air fry',
    'poach', 'braise', 'stew', 'barbecue', 'smoke', 'toast', 'char',
    'sear', 'brown', 'caramelize', 'blacken', 'scorch', 'singe', 'burn',
    'flambe', 'torch', 'griddle', 'blanch', 'parboil', 'parbake',
    'slow cook', 'pressure cook', 'microwave', 'nuke',
    
    # Transformations thermiques passives
    'heat', 'warm', 'reheat', 'preheat', 'cool', 'chill', 'freeze',
    'refrigerate', 'thaw', 'defrost', 'melt', 'soften',
    
    # Processus chimiques/biologiques
    'ferment', 'proof', 'rise', 'leaven', 'activate', 'culture',
    'marinate', 'macerate', 'pickle', 'cure', 'age', 'ripen',
    'caramelize', 'crystallize', 'coagulate', 'curdle', 'congeal',
    'thicken', 'reduce', 'concentrate', 'emulsify', 'dissolve',
    
    # États/résultats passifs
    'rest', 'set', 'settle', 'steep', 'infuse', 'soak',
    'crisp', 'harden', 'firm', 'stiffen', 'soften', 'wilt',
    
    # Appareils qui travaillent seuls
    'blend', 'blenderize', 'blitz', 'grind', 'mill', 'process', 'puree',
    'cream', 'whip', 'churn', 'froth', 'foam',
    
    # Mots non-verbes à supprimer
    'bok', 'cheeses', 'chestnuts', 'chorizo', 'choy', 'ciabatta',
    'cucumber', 'curry', 'plum', 'tons', 'tortillas', 'won',
    'medium high', 'periodically', 'room', 'freezer', 'graham',
    'marzipan', 'frisee', 'fringe', 'degrees', 'minutes', 'seconds',
}


# =============================================================================
# CLASSE PRINCIPALE : VerbCleaner
# =============================================================================

@dataclass
class CleaningStats:
    """Statistiques de nettoyage"""
    total_input: int = 0
    unique_verbs_found: int = 0
    verbs_normalized: int = 0
    verbs_removed: int = 0
    verbs_kept: int = 0
    fuzzy_matches: int = 0
    processing_time: float = 0.0
    
    def __str__(self):
        return f"""
╔══════════════════════════════════════════════════════╗
║           STATISTIQUES DE NETTOYAGE                  ║
╠══════════════════════════════════════════════════════╣
║  Lignes traitées:        {self.total_input:>15,}        ║
║  Verbes uniques trouvés: {self.unique_verbs_found:>15,}        ║
║  Verbes normalisés:      {self.verbs_normalized:>15,}        ║
║  Verbes supprimés:       {self.verbs_removed:>15,}        ║
║  Verbes conservés:       {self.verbs_kept:>15,}        ║
║  Matches fuzzy:          {self.fuzzy_matches:>15,}        ║
║  Temps de traitement:    {self.processing_time:>12.2f} sec   ║
╚══════════════════════════════════════════════════════╝
"""


class VerbCleaner:
    """
    Nettoyeur de verbes culinaires ultra-optimisé.
    
    Combine:
    - Normalisation (variantes → forme canonique)
    - Filtrage (suppression des verbes non-gestuels)
    - Matching flou avec RapidFuzz
    - Cache pour performance maximale
    """
    
    def __init__(
        self,
        normalization_dict: Optional[Dict[str, str]] = None,
        non_gesture_verbs: Optional[Set[str]] = None,
        similarity_threshold: int = 85,
        use_fuzzy: bool = True,
        verbose: bool = True
    ):
        """
        Initialise le cleaner.
        
        Args:
            normalization_dict: Dictionnaire de normalisation (variante → canonique)
            non_gesture_verbs: Set des verbes non-gestuels à filtrer (None = utiliser défaut)
            similarity_threshold: Seuil de similarité pour fuzzy matching (0-100)
            use_fuzzy: Activer le matching flou
            verbose: Afficher les messages de progression
        """
        self.threshold = similarity_threshold
        self.use_fuzzy = use_fuzzy
        self.verbose = verbose
        
        # CORRECTION ICI: Utiliser is None pour distinguer None de set()
        norm_dict = normalization_dict if normalization_dict is not None else MAPPING_VERBES
        remove_set = non_gesture_verbs if non_gesture_verbs is not None else NON_GESTURE_VERBS
        
        # Pré-normaliser les dictionnaires (tout en minuscules, sans tirets)
        self.norm_exact = {
            self._normalize_surface(k): self._normalize_surface(v)
            for k, v in norm_dict.items()
        }
        self.removal_exact = {
            self._normalize_surface(v) for v in remove_set
        }
        
        # Listes pour fuzzy matching
        self.norm_keys = list(self.norm_exact.keys())
        self.removal_list = list(self.removal_exact)
        
        # Caches pour éviter les recalculs
        self._norm_cache: Dict[str, str] = {}
        self._remove_cache: Dict[str, bool] = {}
        self._final_cache: Dict[str, Optional[str]] = {}
        
        # Statistiques
        self.stats = CleaningStats()
        
        if self.verbose:
            print(f"✓ VerbCleaner initialisé:")
            print(f"  - {len(self.norm_exact)} règles de normalisation")
            print(f"  - {len(self.removal_exact)} verbes non-gestuels")
            print(f"  - Seuil fuzzy: {self.threshold}%")
            print(f"  - Fuzzy matching: {'activé' if self.use_fuzzy else 'désactivé'}")
            print(f"  - Mode: {'normalisation seule' if len(self.removal_exact) == 0 else 'normalisation + filtrage'}")
    
    @staticmethod
    def _normalize_surface(text: str) -> str:
        """Normalisation de surface (minuscules, espaces, tirets)"""
        text = str(text).lower().strip()
        text = text.replace("-", " ")
        text = re.sub(r"\s+", " ", text)
        return text
    
    def _normalize_with_dict(self, verb: str) -> str:
        """Normalise un verbe avec le dictionnaire (+ fuzzy si activé)"""
        if verb in self._norm_cache:
            return self._norm_cache[verb]
        
        # 1. Match exact
        if verb in self.norm_exact:
            result = self.norm_exact[verb]
            self._norm_cache[verb] = result
            self.stats.verbs_normalized += 1
            return result
        
        # 2. Fuzzy match si activé
        if self.use_fuzzy and self.norm_keys:
            match_result = process.extractOne(
                verb,
                self.norm_keys,
                scorer=fuzz.ratio,
                score_cutoff=self.threshold
            )
            if match_result is not None:
                matched_key, score, _ = match_result
                result = self.norm_exact[matched_key]
                self._norm_cache[verb] = result
                self.stats.verbs_normalized += 1
                self.stats.fuzzy_matches += 1
                return result
        
        # 3. Pas de match → garder tel quel
        self._norm_cache[verb] = verb
        return verb
    
    def _should_remove(self, verb: str) -> bool:
        """Vérifie si un verbe doit être supprimé (non-gestuel)"""
        # Si pas de verbes à supprimer, ne rien supprimer
        if not self.removal_exact:
            return False
        
        if verb in self._remove_cache:
            return self._remove_cache[verb]
        
        # 1. Match exact
        if verb in self.removal_exact:
            self._remove_cache[verb] = True
            return True
        
        # 2. Fuzzy match si activé
        if self.use_fuzzy and self.removal_list:
            match_result = process.extractOne(
                verb,
                self.removal_list,
                scorer=fuzz.ratio,
                score_cutoff=self.threshold
            )
            if match_result is not None:
                self._remove_cache[verb] = True
                self.stats.fuzzy_matches += 1
                return True
        
        # 3. Pas de match → garder
        self._remove_cache[verb] = False
        return False
    
    def clean_verb(self, verb: str) -> Optional[str]:
        """
        Nettoie un seul verbe: normalise puis filtre.
        
        Returns:
            Verbe normalisé ou None si à supprimer
        """
        # Check cache final
        if verb in self._final_cache:
            return self._final_cache[verb]
        
        # Normalisation de surface
        verb_norm = self._normalize_surface(verb)
        
        # Normalisation avec dictionnaire
        verb_norm = self._normalize_with_dict(verb_norm)
        
        # Filtrage
        if self._should_remove(verb_norm):
            self._final_cache[verb] = None
            self.stats.verbs_removed += 1
            return None
        
        self._final_cache[verb] = verb_norm
        self.stats.verbs_kept += 1
        return verb_norm
    
    def clean_list(self, verbs: Union[str, List[str]]) -> List[str]:
        """
        Nettoie une liste de verbes.
        
        Args:
            verbs: Liste de verbes (str format liste ou List[str])
            
        Returns:
            Liste de verbes nettoyés (normalisés + filtrés)
        """
        # Conversion si chaîne
        if isinstance(verbs, str):
            try:
                verbs = ast.literal_eval(verbs)
            except (ValueError, SyntaxError):
                verbs = [verbs]
        
        if not isinstance(verbs, list):
            return []
        
        # Nettoyage
        cleaned = []
        for verb in verbs:
            result = self.clean_verb(verb)
            if result is not None:
                cleaned.append(result)
        
        return cleaned
    
    def clean_dataframe(
        self, 
        df, 
        input_col: str = 'actions',
        output_col: str = 'actions_cleaned'
    ):
        """
        Nettoie une colonne de DataFrame (optimisé).
        
        Args:
            df: DataFrame pandas
            input_col: Nom de la colonne d'entrée
            output_col: Nom de la colonne de sortie
            
        Returns:
            DataFrame avec nouvelle colonne nettoyée
        """
        if not HAS_PANDAS:
            raise ImportError("pandas requis pour cette fonction")
        
        start_time = time.time()
        self.stats = CleaningStats()  # Reset stats
        self.stats.total_input = len(df)
        
        if self.verbose:
            print(f"\n🚀 Nettoyage du DataFrame...")
            print(f"   {len(df):,} lignes à traiter")
        
        # Étape 1: Extraire tous les verbes uniques
        if self.verbose:
            print("\n📋 Étape 1/3: Extraction des verbes uniques...")
        
        all_verbs = set()
        for actions_list in df[input_col]:
            if isinstance(actions_list, str):
                try:
                    actions_list = ast.literal_eval(actions_list)
                except:
                    actions_list = [actions_list]
            if isinstance(actions_list, list):
                all_verbs.update(actions_list)
        
        self.stats.unique_verbs_found = len(all_verbs)
        if self.verbose:
            print(f"   ✓ {len(all_verbs):,} verbes uniques trouvés")
        
        # Étape 2: Pré-calculer le mapping pour tous les verbes uniques
        if self.verbose:
            print("\n🧹 Étape 2/3: Pré-calcul du mapping...")
        
        verb_mapping = {}
        for verb in all_verbs:
            result = self.clean_verb(verb)
            if result is not None:
                verb_mapping[verb] = result
        
        if self.verbose:
            print(f"   ✓ {len(verb_mapping):,} verbes conservés")
            print(f"   ✓ {len(all_verbs) - len(verb_mapping):,} verbes supprimés")
        
        # Étape 3: Appliquer le mapping (ultra-rapide!)
        if self.verbose:
            print("\n⚡ Étape 3/3: Application du mapping...")
        
        def apply_mapping(actions_list):
            if isinstance(actions_list, str):
                try:
                    actions_list = ast.literal_eval(actions_list)
                except:
                    actions_list = [actions_list]
            if not isinstance(actions_list, list):
                return []
            return [verb_mapping[v] for v in actions_list if v in verb_mapping]
        
        df_result = df.copy()
        df_result[output_col] = df[input_col].apply(apply_mapping)
        
        # Finaliser stats
        self.stats.processing_time = time.time() - start_time
        
        if self.verbose:
            print(self.stats)
            
            # Stats supplémentaires
            avg_before = df[input_col].apply(
                lambda x: len(ast.literal_eval(x) if isinstance(x, str) else x) 
                if x else 0
            ).mean()
            avg_after = df_result[output_col].apply(len).mean()
            
            print(f"📊 Réduction moyenne: {avg_before:.2f} → {avg_after:.2f} actions/recette")
            if avg_before > 0:
                print(f"   ({(1 - avg_after/avg_before)*100:.1f}% de réduction)")
        
        return df_result
    
    def clear_cache(self):
        """Vide tous les caches"""
        self._norm_cache.clear()
        self._remove_cache.clear()
        self._final_cache.clear()
        if self.verbose:
            print("✓ Caches vidés")
    
    def get_cache_stats(self) -> Dict:
        """Retourne les statistiques des caches"""
        return {
            'norm_cache_size': len(self._norm_cache),
            'remove_cache_size': len(self._remove_cache),
            'final_cache_size': len(self._final_cache),
        }



# =============================================================================
# FONCTIONS UTILITAIRES
# =============================================================================

def load_verbs_from_file(filepath: str) -> Set[str]:
    """Charge une liste de verbes depuis un fichier texte"""
    verbs = set()
    path = Path(filepath)
    
    if not path.exists():
        raise FileNotFoundError(f"Fichier non trouvé: {filepath}")
    
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                verbs.add(line.lower())
    
    return verbs


def create_cleaner_from_files(
    normalization_file: Optional[str] = None,
    non_gesture_file: Optional[str] = None,
    **kwargs
) -> VerbCleaner:
    """
    Crée un VerbCleaner à partir de fichiers externes.
    
    Args:
        normalization_file: Chemin vers fichier de normalisation (format: variante,canonique)
        non_gesture_file: Chemin vers fichier des verbes non-gestuels
        **kwargs: Arguments supplémentaires pour VerbCleaner
    """
    norm_dict = None
    non_gesture = None
    
    if normalization_file:
        norm_dict = {}
        with open(normalization_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#') and ',' in line:
                    parts = line.split(',', 1)
                    if len(parts) == 2:
                        norm_dict[parts[0].strip()] = parts[1].strip()
    
    if non_gesture_file:
        non_gesture = load_verbs_from_file(non_gesture_file)
    
    return VerbCleaner(
        normalization_dict=norm_dict,
        non_gesture_verbs=non_gesture,
        **kwargs
    )




In [13]:
"""
EXEMPLES D'UTILISATION DU VERBCLEANER
=====================================
Utilisation directe avec les variables mapping_verbes et non_gesture_verbs
"""





# Créer un cleaner qui ne filtre PAS (non_gesture_verbs vide)
cleaner_norm_only = VerbCleaner(
    normalization_dict=MAPPING_VERBES,
    non_gesture_verbs=set(),  # ← Set vide = pas de filtrage
    similarity_threshold=95,
    use_fuzzy=True,
    verbose=True
)
df =  pd.read_csv("data/final_dataset_cleaned_with_non_gestures.csv",index_col=0)
df = df[['id','title','actions','type','type_2']]

# Appliquer sur le DataFrame
df_normalized = cleaner_norm_only.clean_dataframe(
    df, 
    input_col='actions', 
    output_col='actions_normalized'
)









✓ VerbCleaner initialisé:
  - 241 règles de normalisation
  - 0 verbes non-gestuels
  - Seuil fuzzy: 95%
  - Fuzzy matching: activé
  - Mode: normalisation seule

🚀 Nettoyage du DataFrame...
   2,749,491 lignes à traiter

📋 Étape 1/3: Extraction des verbes uniques...
   ✓ 594 verbes uniques trouvés

🧹 Étape 2/3: Pré-calcul du mapping...
   ✓ 594 verbes conservés
   ✓ 0 verbes supprimés

⚡ Étape 3/3: Application du mapping...

╔══════════════════════════════════════════════════════╗
║           STATISTIQUES DE NETTOYAGE                  ║
╠══════════════════════════════════════════════════════╣
║  Lignes traitées:              2,749,491        ║
║  Verbes uniques trouvés:             594        ║
║  Verbes normalisés:                  101        ║
║  Verbes supprimés:                     0        ║
║  Verbes conservés:                   594        ║
║  Matches fuzzy:                        0        ║
║  Temps de traitement:          484.93 sec   ║
╚══════════════════════════════════════

In [8]:
recipes_df = pd.read_csv("strategy_2_results/recipes_enriched.csv")
recipes_df

,id,title,url,partition,number_of_ingredients,number_of_steps,category,complexity
0,000018c8a5,Worlds Best Mac and Cheese,http://www.epicurious.com/recipes/food/views/-...,train,14,23,other,elevee
1,000033e39b,Dilly Macaroni Salad Recipe,http://cookeatshare.com/recipes/dilly-macaroni...,train,9,8,bakery,simple
2,000035f7ed,Gazpacho,http://www.foodnetwork.com/recipes/gazpacho1.html,train,9,4,other,simple
3,00003a70b1,Crunchy Onion Potato Bake,http://www.food.com/recipe/crunchy-onion-potat...,test,7,9,other,simple
4,00004320bb,Cool 'n Easy Creamy Watermelon Pie,http://www.food.com/recipe/cool-n-easy-creamy-...,train,5,7,bakery,simple
...,...,...,...,...,...,...,...,...
1029715,ffffbb45d2,Baumkuchen -- the King of Cakes!,http://www.food.com/recipe/baumkuchen-the-king...,val,15,11,bakery,moyenne
1029716,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,https://cookpad.com/us/recipes/153324-extremel...,train,5,3,quick_prep,simple
1029717,ffffd33513,Pan-Roasted Pork Chops With Apple Fritters,http://cooking.nytimes.com/recipes/1015164,train,28,22,other,elevee
1029718,ffffd533d7,Polpette in Spicy Tomato Sauce,http://www.foodandwine.com/recipes/polpette-sp...,train,12,7,other,simple


In [ ]:
r

In [4]:

# Créer un cleaner qui ne filtre PAS (non_gesture_verbs vide)
cleaner_norm_only = VerbCleaner(
    normalization_dict=MAPPING_VERBES,
    non_gesture_verbs= NON_GESTURE_VERBS,  # ← Set vide = pas de filtrage
    similarity_threshold=95,
    use_fuzzy=True,
    verbose=True
)
df1 =  pd.read_csv("data/final_dataset_cleaned_with_non_gestures.csv",index_col=0)
df1 = df1[['id','title','actions','type','type_2']]

# Appliquer sur le DataFrame
df1_normalized = cleaner_norm_only.clean_dataframe(
    df1, 
    input_col='actions', 
    output_col='actions_normalized'
)

✓ VerbCleaner initialisé:
  - 241 règles de normalisation
  - 116 verbes non-gestuels
  - Seuil fuzzy: 95%
  - Fuzzy matching: activé
  - Mode: normalisation + filtrage

🚀 Nettoyage du DataFrame...
   2,749,491 lignes à traiter

📋 Étape 1/3: Extraction des verbes uniques...
   ✓ 594 verbes uniques trouvés

🧹 Étape 2/3: Pré-calcul du mapping...
   ✓ 494 verbes conservés
   ✓ 100 verbes supprimés

⚡ Étape 3/3: Application du mapping...

╔══════════════════════════════════════════════════════╗
║           STATISTIQUES DE NETTOYAGE                  ║
╠══════════════════════════════════════════════════════╣
║  Lignes traitées:              2,749,491        ║
║  Verbes uniques trouvés:             594        ║
║  Verbes normalisés:                  101        ║
║  Verbes supprimés:                   100        ║
║  Verbes conservés:                   494        ║
║  Matches fuzzy:                        0        ║
║  Temps de traitement:          146.14 sec   ║
╚═════════════════════════════

In [5]:
df1_normalized

,id,title,actions,type,type_2,actions_normalized
Unnamed: 0,,,,,,
37807,000018c8a5,Worlds Best Mac and Cheese,"['preheat', 'butter', 'cook', 'clean', 'place'...",principal,variante_principale,"[butter, cook, clean, place, mix, scrape, spri..."
12667,000033e39b,Dilly Macaroni Salad Recipe,"['cook', 'drain', 'mix', 'blend', 'add', 'toss...",principal,variante_principale,"[cook, drain, mix, add, toss, serve]"
12668,000033e39b,Dilly Macaroni Salad Recipe,"['cook', 'drain', 'cube', 'slice', 'mince', 'm...",secondaire,variante_ingredients,"[cook, drain, cube, slice, mince, mix, add, to..."
15831,000035f7ed,Gazpacho,"['puree', 'mix', 'chill', 'drizzle', 'garnish']",principal,variante_principale,"[mix, drizzle, garnish]"
15832,000035f7ed,Gazpacho,"['quarter', 'cut', 'puree', 'mix', 'chop', 'ch...",secondaire,variante_ingredients,"[quarter, cut, mix, chop, drizzle, garnish]"
...,...,...,...,...,...,...
4057025,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"['julienne', 'squeeze', 'serve', 'mix']",secondaire,variante_permutation,"[julienne, squeeze, serve, mix]"
4057026,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"['julienne', 'mix', 'squeeze', 'serve']",secondaire,variante_permutation,"[julienne, mix, squeeze, serve]"
4062663,ffffdea29a,Mexican-Style Sweetened Black Coffee (Cafe de ...,"['boil', 'stir', 'add', 'boil', 'strain', 'rem...",principal,variante_principale,"[stir, add, drain, remove]"


In [6]:
df1_normalized.reset_index(drop=True, inplace=True)
df1_normalized['actions'] = df1_normalized['actions_normalized']
df1_normalized = df1_normalized.drop(columns=['actions_normalized'])
df1_normalized.to_csv("data/final_dataset_cleaned_with_normalization_and_filtering.csv")
df1_normalized

,id,title,actions,type,type_2
0,000018c8a5,Worlds Best Mac and Cheese,"[butter, cook, clean, place, mix, scrape, spri...",principal,variante_principale
1,000033e39b,Dilly Macaroni Salad Recipe,"[cook, drain, mix, add, toss, serve]",principal,variante_principale
2,000033e39b,Dilly Macaroni Salad Recipe,"[cook, drain, cube, slice, mince, mix, add, to...",secondaire,variante_ingredients
3,000035f7ed,Gazpacho,"[mix, drizzle, garnish]",principal,variante_principale
4,000035f7ed,Gazpacho,"[quarter, cut, mix, chop, drizzle, garnish]",secondaire,variante_ingredients
...,...,...,...,...,...
2749486,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"[julienne, squeeze, serve, mix]",secondaire,variante_permutation
2749487,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"[julienne, mix, squeeze, serve]",secondaire,variante_permutation
2749488,ffffdea29a,Mexican-Style Sweetened Black Coffee (Cafe de ...,"[stir, add, drain, remove]",principal,variante_principale
2749489,ffffdea29a,Mexican-Style Sweetened Black Coffee (Cafe de ...,"[stir, add, drain, remove]",secondaire,variante_ingredients


In [10]:
 # 5. Supprimer les doublons consécutifs 
print("→ Suppression des doublons consécutifs...")
final_df=  df1_normalized.copy()
final_df['actions'] = final_df['actions'].apply(
            lambda x: remove_consecutive_duplicates(x) if isinstance(x, (list, np.ndarray)) else x
        )
       
        # 6. Supprimer les doublons de listes d'actions identiques pour une même recette
print("→ Suppression des doublons par ID...")
        
        # Pour final_df
final_df['actions_tuple'] = final_df['actions'].apply(
            lambda x: tuple(x) if isinstance(x, (list, np.ndarray)) else x
        )
before_dedup = len(final_df)
final_df = final_df.drop_duplicates(subset=['id', 'actions_tuple'], keep='first')
final_df = final_df.drop(columns=['actions_tuple'])
print(f"  ✓ final_df: {before_dedup - len(final_df):,} doublons supprimés ({len(final_df):,} restantes)")
        
        

→ Suppression des doublons consécutifs...
→ Suppression des doublons par ID...
  ✓ final_df: 441,336 doublons supprimés (2,308,155 restantes)


In [20]:
if __name__ == "__main__":
 
    
    # Chemin du fichier
    GRAPHS_RECIPES_CSV = "data/final_dataset_cleaned_with_normalization_only.csv"
    
    # Exécuter le pipeline
    run_strategy_3_pipeline(
        graphs_recipes_csv=GRAPHS_RECIPES_CSV,
        output_dir="strategy_3_results"
    )


################################################################################
# PIPELINE STRATÉGIE 3 - DÉTECTION DES SUCCESSIONS ILLOGIQUES
################################################################################

ÉTAPE 1/5 : Export de la taxonomie des verbes...
✅ Taxonomie exportée : strategy_3_results\verb_taxonomy.json
📊 Total verbes : 681

ÉTAPE 2/5 : Chargement des données...
  ✅ Graphes chargés : 2,749,491

ÉTAPE 3/5 : Analyse et détection des violations...

ANALYSE DU DATASET - DÉTECTION DES VIOLATIONS

Traitement de 2,749,491 graphes...

  Progression : 10,000/2,749,491 (0.4%)
  Progression : 20,000/2,749,491 (0.7%)
  Progression : 30,000/2,749,491 (1.1%)
  Progression : 40,000/2,749,491 (1.5%)
  Progression : 50,000/2,749,491 (1.8%)
  Progression : 60,000/2,749,491 (2.2%)
  Progression : 70,000/2,749,491 (2.5%)
  Progression : 80,000/2,749,491 (2.9%)
  Progression : 90,000/2,749,491 (3.3%)
  Progression : 100,000/2,749,491 (3.6%)
  Progression : 110,000/2,749,491 (

In [ ]:
 =============================================================================
# EXEMPLE 2: NORMALISATION + FILTRAGE (pipeline complet)
# =============================================================================

print("=" * 70)
print("EXEMPLE 2: NORMALISATION + FILTRAGE")
print("=" * 70)
print("→ Les verbes sont standardisés ET les non-gestes sont supprimés")
print()

# Créer un cleaner complet (normalisation + filtrage)
cleaner_full = VerbCleaner(
    normalization_dict=MAPPING_VERBES,
    non_gesture_verbs=NON_GESTURE_VERBS,  # ← Filtrage activé
    similarity_threshold=85,
    use_fuzzy=True,
    verbose=False
)

# Appliquer sur le DataFrame
df_cleaned = cleaner_full.clean_dataframe(
    df_test, 
    input_col='actions', 
    output_col='actions_cleaned'
)

print("Résultat (normalisation + filtrage):")
print("-" * 70)
for _, row in df_cleaned.iterrows():
    original = ast.literal_eval(row['actions'])
    cleaned = row['actions_cleaned']
    removed = [a for a in original if a.lower() not in [c for c in cleaned]]
    print(f"Recipe {row['recipe_id']} ({row['recipe_name']}):")
    print(f"  AVANT:    {original}")
    print(f"  APRÈS:    {cleaned}")
    print(f"  SUPPRIMÉS: {removed}")
    print()


# =============================================================================
# EXEMPLE 3: COMPARAISON CÔTE À CÔTE
# =============================================================================

print("=" * 70)
print("EXEMPLE 3: COMPARAISON CÔTE À CÔTE")
print("=" * 70)

# Fusionner les résultats
df_compare = df_test.copy()
df_compare['normalized'] = df_normalized['actions_normalized']
df_compare['cleaned'] = df_cleaned['actions_cleaned']

print(f"\n{'Recipe':<10} {'Original':<45} {'Normalized':<35} {'Cleaned (gestes)':<25}")
print("-" * 115)

for _, row in df_compare.iterrows():
    orig = str(ast.literal_eval(row['actions']))[:42]
    norm = str(row['normalized'])[:32]
    clean = str(row['cleaned'])[:22]
    print(f"{row['recipe_name']:<10} {orig:<45} {norm:<35} {clean:<25}")

In [3]:
graph_nettoye

,Unnamed: 0.1,Unnamed: 0,id,title,actions,type,type_2
0,0,37807,000018c8a5,Worlds Best Mac and Cheese,"['preheat', 'butter', 'cook', 'clean', 'place'...",principal,variante_principale
1,1,12667,000033e39b,Dilly Macaroni Salad Recipe,"['cook', 'drain', 'mix', 'blend', 'add', 'toss...",principal,variante_principale
2,2,12668,000033e39b,Dilly Macaroni Salad Recipe,"['cook', 'drain', 'cube', 'slice', 'mince', 'm...",secondaire,variante_ingredients
3,3,15831,000035f7ed,Gazpacho,"['puree', 'mix', 'chill', 'drizzle', 'garnish']",principal,variante_principale
4,4,15832,000035f7ed,Gazpacho,"['quarter', 'cut', 'puree', 'mix', 'chop', 'ch...",secondaire,variante_ingredients
...,...,...,...,...,...,...,...
2749486,2749486,4057025,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"['julienne', 'squeeze', 'serve', 'mix']",secondaire,variante_permutation
2749487,2749487,4057026,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"['julienne', 'mix', 'squeeze', 'serve']",secondaire,variante_permutation
2749488,2749488,4062663,ffffdea29a,Mexican-Style Sweetened Black Coffee (Cafe de ...,"['boil', 'stir', 'add', 'boil', 'strain', 'rem...",principal,variante_principale
2749489,2749489,4062664,ffffdea29a,Mexican-Style Sweetened Black Coffee (Cafe de ...,"['grind', 'boil', 'stir', 'add', 'boil', 'stra...",secondaire,variante_ingredients


In [ ]:
data_test = graph_nettoye[graph_nettoye["id"].isin(ids_test3) & (graph_nettoye["type_2"] == "variante_ingredients")]
data
#data_test2.merge(recipes_df[['id','number_of_steps']], on="id")


,Unnamed: 0,id,title,actions,type,type_2
0,37807,000018c8a5,Worlds Best Mac and Cheese,"[butter, clean, place, mix, scrape, sprinkle, ...",principal,variante_principale
1,12667,000033e39b,Dilly Macaroni Salad Recipe,"[drain, mix, add, toss]",principal,variante_principale
2,12668,000033e39b,Dilly Macaroni Salad Recipe,"[drain, cube, slice, mince, mix, add, toss]",secondaire,variante_ingredients
3,15831,000035f7ed,Gazpacho,"[mix, drizzle, garnish]",principal,variante_principale
4,15832,000035f7ed,Gazpacho,"[quarter, cut, mix, chop, drizzle, garnish]",secondaire,variante_ingredients
...,...,...,...,...,...,...
2241612,4050067,ffffbb45d2,Baumkuchen -- the King of Cakes!,"[whip, mix, beat, fill, repeat, cut, remove, f...",principal,variante_principale
2241613,4057023,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"[julienne, squeeze, mix]",principal,variante_principale
2241614,4057026,ffffcd4444,Extremely Easy and Quick - Namul Daikon Salad,"[julienne, mix, squeeze]",secondaire,variante_permutation
2241615,4062663,ffffdea29a,Mexican-Style Sweetened Black Coffee (Cafe de ...,"[stir, add, strain, remove]",principal,variante_principale


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from itertools import combinations

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10

def run_eda_pipeline(df_graphs, df_metadata, output_dir='C:/Users/nguem/Downloads/recipes-20250605T021334Z-1-002/recipes/EDA_outputs'):
    """
    Pipeline EDA pour analyser les graphes de recettes
    
    Args:
        df_graphs: DataFrame avec ['id','title','actions','type','type_2']
        df_metadata: DataFrame avec ['id','title','category','complexity','number_of_steps']
    """
    
    # Merge des dataframes
    df = df_graphs.merge(df_metadata, on=['id', 'title'], how='inner')
    df['num_actions'] = df['actions'].apply(len)
    df['num_unique_actions'] = df['actions'].apply(lambda x: len(set(x)))
    
    # ========================================================================
    # GRAPHIQUE 1: Distribution longueur des séquences
    # ========================================================================
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(df['num_actions'], bins=50, color='#6A5ACD', alpha=0.7, edgecolor='black')
    ax.axvline(df['num_actions'].mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {df["num_actions"].mean():.1f}')
    ax.axvline(df['num_actions'].median(), color='orange', linestyle='--', linewidth=2, label=f'Médiane: {df["num_actions"].median():.1f}')
    ax.set_xlabel('Nombre d\'actions', fontsize=12, fontweight='bold')
    ax.set_ylabel('Fréquence', fontsize=12, fontweight='bold')
    ax.set_title('Distribution de la longueur des séquences d\'actions', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph1_distribution_longueur.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 2: Distribution par catégorie
    # ========================================================================
    fig, ax = plt.subplots(figsize=(10, 6))
    category_counts = df['category'].value_counts().sort_values()
    colors = sns.color_palette("rocket", len(category_counts))
    bars = ax.barh(category_counts.index, category_counts.values, color=colors, edgecolor='black', linewidth=1.5)
    ax.set_xlabel('Nombre de recettes', fontsize=12, fontweight='bold')
    ax.set_ylabel('Catégorie', fontsize=12, fontweight='bold')
    ax.set_title('Distribution des recettes par catégorie de cuisine', fontsize=14, fontweight='bold')
    for i, (bar, count) in enumerate(zip(bars, category_counts.values)):
        ax.text(count + max(category_counts)*0.01, i, f'{count:,}', va='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph2_distribution_categorie.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 3: Heatmap longueur par catégorie × complexité
    # ========================================================================
    pivot = df.groupby(['category', 'complexity'])['num_actions'].mean().unstack(fill_value=0)
    pivot = pivot[['simple', 'moyenne', 'elevee']].T  # Transpose pour format vertical
    
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Longueur moyenne'}, 
                linewidths=1.5, linecolor='black', ax=ax, annot_kws={'fontsize': 11, 'fontweight': 'bold'})
    ax.set_xlabel('Catégorie de cuisine', fontsize=12, fontweight='bold')
    ax.set_ylabel('Complexité', fontsize=12, fontweight='bold')
    ax.set_title('Longueur moyenne des séquences par catégorie et complexité', fontsize=14, fontweight='bold')
    ax.set_yticklabels(['Simple', 'Moyenne', 'Élevée'], rotation=0)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph3_heatmap_categorie_complexite.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 4: Top 20 gestes les plus fréquents
    # ========================================================================
    all_actions = [action for actions_list in df['actions'] for action in actions_list]
    action_counts = Counter(all_actions).most_common(20)
    actions, counts = zip(*action_counts)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette("viridis", 20)
    bars = ax.barh(actions, counts, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_xlabel('Nombre d\'occurrences', fontsize=12, fontweight='bold')
    ax.set_ylabel('Geste culinaire', fontsize=12, fontweight='bold')
    ax.set_title('Top 20 des gestes culinaires les plus fréquents', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    for i, (bar, count) in enumerate(zip(bars, counts)):
        ax.text(count + max(counts)*0.01, i, f'{count:,}', va='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph4_top20_gestes.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 5: Boxplot number_of_steps par complexité
    # ========================================================================
    fig, ax = plt.subplots(figsize=(10, 6))
    complexity_order = ['simple', 'moyenne', 'elevee']
    df_sorted = df[df['complexity'].isin(complexity_order)]
    
    parts = ax.violinplot([df_sorted[df_sorted['complexity']==c]['number_of_steps'].values 
                            for c in complexity_order], 
                           positions=[0, 1, 2], showmeans=True, showmedians=True)
    
    for pc in parts['bodies']:
        pc.set_facecolor('#FF6B6B')
        pc.set_alpha(0.6)
    
    ax.boxplot([df_sorted[df_sorted['complexity']==c]['number_of_steps'].values 
                for c in complexity_order], 
               positions=[0, 1, 2], widths=0.3)
    
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Simple', 'Moyenne', 'Élevée'], fontsize=11)
    ax.set_ylabel('Nombre d\'étapes (number_of_steps)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Complexité', fontsize=12, fontweight='bold')
    ax.set_title('Distribution du nombre d\'étapes par complexité', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph5_boxplot_steps_complexity.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 6: Matrice co-occurrence Top 15 gestes
    # ========================================================================
    top15_actions = [a for a, c in action_counts[:15]]
    cooccurrence = np.zeros((15, 15))
    
    for actions_list in df['actions']:
        actions_set = [a for a in actions_list if a in top15_actions]
        for a1, a2 in combinations(set(actions_set), 2):
            i1, i2 = top15_actions.index(a1), top15_actions.index(a2)
            cooccurrence[i1][i2] += 1
            cooccurrence[i2][i1] += 1
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cooccurrence, xticklabels=top15_actions, yticklabels=top15_actions, 
                cmap='Blues', annot=False, fmt='g', cbar_kws={'label': 'Co-occurrences'}, 
                linewidths=0.5, ax=ax)
    ax.set_title('Matrice de co-occurrence des Top 15 gestes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph6_cooccurrence_matrix.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 7: Top 15 transitions (bigrams)
    # ========================================================================
    bigrams = []
    for actions_list in df['actions']:
        for i in range(len(actions_list) - 1):
            bigrams.append(f"{actions_list[i]} → {actions_list[i+1]}")
    
    bigram_counts = Counter(bigrams).most_common(15)
    transitions, counts = zip(*bigram_counts)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette("rocket_r", 15)
    bars = ax.barh(transitions, counts, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_xlabel('Fréquence', fontsize=12, fontweight='bold')
    ax.set_ylabel('Transition', fontsize=12, fontweight='bold')
    ax.set_title('Top 15 des transitions de gestes les plus fréquentes', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    for i, (bar, count) in enumerate(zip(bars, counts)):
        ax.text(count + max(counts)*0.01, i, f'{count:,}', va='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph7_top15_transitions.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 8: Scatter longueur × diversité
    # ========================================================================
    fig, ax = plt.subplots(figsize=(12, 8))
    scatter = ax.hexbin(df['num_actions'], df['num_unique_actions'], gridsize=30, 
                        cmap='YlOrRd', mincnt=1, edgecolors='black', linewidths=0.2)
    ax.plot([0, df['num_actions'].max()], [0, df['num_actions'].max()], 
            'r--', linewidth=2, label='Diversité maximale (pas de répétition)')
    ax.set_xlabel('Longueur de séquence (nombre total d\'actions)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Nombre de gestes uniques', fontsize=12, fontweight='bold')
    ax.set_title('Diversité du vocabulaire gestuel vs longueur de séquence', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Densité de recettes', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph8_diversite_vocabulaire.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 9: Comparaison gestes vs étapes totales
    # ========================================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Scatter plot
    axes[0].scatter(df['number_of_steps'], df['num_actions'], alpha=0.3, c='#4ECDC4', edgecolors='black', linewidths=0.5, s=20)
    axes[0].plot([0, df['number_of_steps'].max()], [0, df['number_of_steps'].max()], 
                 'r--', linewidth=2, label='Égalité (gestes = étapes)')
    axes[0].set_xlabel('Nombre d\'étapes totales (number_of_steps)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Nombre de gestes (len(actions))', fontsize=12, fontweight='bold')
    axes[0].set_title('Comparaison: Gestes extraits vs Étapes totales', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3)
    
    # Histogramme des différences
    df['diff_steps_actions'] = df['number_of_steps'] - df['num_actions']
    axes[1].hist(df['diff_steps_actions'], bins=50, color='#FF6B6B', alpha=0.7, edgecolor='black')
    axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Égalité')
    axes[1].axvline(df['diff_steps_actions'].mean(), color='orange', linestyle='--', linewidth=2, 
                    label=f'Moyenne: {df["diff_steps_actions"].mean():.1f}')
    axes[1].set_xlabel('Différence (étapes - gestes)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Fréquence', fontsize=12, fontweight='bold')
    axes[1].set_title('Distribution des différences', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=10)
    
    plt.suptitle('Évaluation du nombre de gestes extraits par recette', fontsize=15, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph9_gestes_vs_steps.png', bbox_inches='tight')
    plt.close()
    
    print(f"✅ {9} graphiques EDA générés avec succès dans {output_dir}/")
    
    # Statistiques résumées
    print("\n" + "="*80)
    print("STATISTIQUES RÉSUMÉES")
    print("="*80)
    print(f"Total recettes: {len(df):,}")
    print(f"Longueur moyenne actions: {df['num_actions'].mean():.2f} ± {df['num_actions'].std():.2f}")
    print(f"Nombre moyen étapes: {df['number_of_steps'].mean():.2f} ± {df['number_of_steps'].std():.2f}")
    print(f"Différence moyenne (étapes - gestes): {df['diff_steps_actions'].mean():.2f}")
    print(f"Gestes uniques totaux: {len(set(all_actions))}")
    print("="*80)


if __name__ == "__main__":
    # Exemple d'utilisation (à adapter avec tes vrais dataframes)
    # df_graphs = pd.read_csv('path_to_graphs.csv')
    # df_metadata = pd.read_csv('path_to_metadata.csv')
    run_eda_pipeline(final_df, recipes_df)

✅ 9 graphiques EDA générés avec succès dans C:/Users/nguem/Downloads/recipes-20250605T021334Z-1-002/recipes/EDA_outputs/

STATISTIQUES RÉSUMÉES
Total recettes: 2,308,155
Longueur moyenne actions: 8.59 ± 4.94
Nombre moyen étapes: 10.49 ± 6.68
Différence moyenne (étapes - gestes): 1.91
Gestes uniques totaux: 406


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from itertools import combinations

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10

def run_eda_pipeline(df_graphs, df_metadata, output_dir='C:/Users/nguem/Downloads/recipes-20250605T021334Z-1-002/recipes/EDA_outputs'):
    """
    Pipeline EDA pour analyser les graphes de recettes
    
    Args:
        df_graphs: DataFrame avec ['id','title','actions','type','type_2']
        df_metadata: DataFrame avec ['id','title','category','complexity','number_of_steps']
    """
    os.makedirs(output_dir, exist_ok=True)
    # Merge des dataframes
    df = df_graphs.merge(df_metadata, on=['id', 'title'], how='inner')
    df['num_actions'] = df['actions'].apply(len)
    df['num_unique_actions'] = df['actions'].apply(lambda x: len(set(x)))
    
    # ========================================================================
    # GRAPHIQUE 1: Distribution longueur des séquences
    # ========================================================================
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(df['num_actions'], bins=50, color='#6A5ACD', alpha=0.7, edgecolor='black')
    ax.axvline(df['num_actions'].mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne: {df["num_actions"].mean():.1f}')
    ax.axvline(df['num_actions'].median(), color='orange', linestyle='--', linewidth=2, label=f'Médiane: {df["num_actions"].median():.1f}')
    ax.set_xlabel('Nombre d\'actions', fontsize=12, fontweight='bold')
    ax.set_ylabel('Fréquence', fontsize=12, fontweight='bold')
    ax.set_title('Distribution de la longueur des séquences d\'actions', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph1_distribution_longueur.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 2: Distribution par catégorie
    # ========================================================================
    fig, ax = plt.subplots(figsize=(10, 6))
    category_counts = df['category'].value_counts().sort_values()
    colors = sns.color_palette("rocket", len(category_counts))
    bars = ax.barh(category_counts.index, category_counts.values, color=colors, edgecolor='black', linewidth=1.5)
    ax.set_xlabel('Nombre de recettes', fontsize=12, fontweight='bold')
    ax.set_ylabel('Catégorie', fontsize=12, fontweight='bold')
    ax.set_title('Distribution des recettes par catégorie de cuisine', fontsize=14, fontweight='bold')
    for i, (bar, count) in enumerate(zip(bars, category_counts.values)):
        ax.text(count + max(category_counts)*0.01, i, f'{count:,}', va='center', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph2_distribution_categorie.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 3: Heatmap longueur par catégorie × complexité
    # ========================================================================
    pivot = df.groupby(['category', 'complexity'])['num_actions'].mean().unstack(fill_value=0)
    pivot = pivot[['simple', 'moyenne', 'elevee']]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Longueur moyenne'}, 
                linewidths=1, linecolor='black', ax=ax)
    ax.set_xlabel('Complexité', fontsize=12, fontweight='bold')
    ax.set_ylabel('Catégorie', fontsize=12, fontweight='bold')
    ax.set_title('Longueur moyenne des séquences par catégorie et complexité', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph3_heatmap_categorie_complexite.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 4: Top 20 gestes les plus fréquents
    # ========================================================================
    all_actions = [action for actions_list in df['actions'] for action in actions_list]
    action_counts = Counter(all_actions).most_common(20)
    actions, counts = zip(*action_counts)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette("viridis", 20)
    bars = ax.barh(actions, counts, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_xlabel('Nombre d\'occurrences', fontsize=12, fontweight='bold')
    ax.set_ylabel('Geste culinaire', fontsize=12, fontweight='bold')
    ax.set_title('Top 20 des gestes culinaires les plus fréquents', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    for i, (bar, count) in enumerate(zip(bars, counts)):
        ax.text(count + max(counts)*0.01, i, f'{count:,}', va='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph4_top20_gestes.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 5: Boxplot number_of_steps par complexité
    # ========================================================================
    fig, ax = plt.subplots(figsize=(10, 6))
    complexity_order = ['simple', 'moyenne', 'elevee']
    df_sorted = df[df['complexity'].isin(complexity_order)]
    
    parts = ax.violinplot([df_sorted[df_sorted['complexity']==c]['number_of_steps'].values 
                            for c in complexity_order], 
                           positions=[0, 1, 2], showmeans=True, showmedians=True)
    
    for pc in parts['bodies']:
        pc.set_facecolor('#FF6B6B')
        pc.set_alpha(0.6)
    
    ax.boxplot([df_sorted[df_sorted['complexity']==c]['number_of_steps'].values 
                for c in complexity_order], 
               positions=[0, 1, 2], widths=0.3)
    
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Simple', 'Moyenne', 'Élevée'], fontsize=11)
    ax.set_ylabel('Nombre d\'étapes (number_of_steps)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Complexité', fontsize=12, fontweight='bold')
    ax.set_title('Distribution du nombre d\'étapes par complexité', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph5_boxplot_steps_complexity.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 6: Matrice co-occurrence Top 15 gestes
    # ========================================================================
    top15_actions = [a for a, c in action_counts[:15]]
    cooccurrence = np.zeros((15, 15))
    
    for actions_list in df['actions']:
        actions_set = [a for a in actions_list if a in top15_actions]
        for a1, a2 in combinations(set(actions_set), 2):
            i1, i2 = top15_actions.index(a1), top15_actions.index(a2)
            cooccurrence[i1][i2] += 1
            cooccurrence[i2][i1] += 1
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cooccurrence, xticklabels=top15_actions, yticklabels=top15_actions, 
                cmap='Blues', annot=False, fmt='g', cbar_kws={'label': 'Co-occurrences'}, 
                linewidths=0.5, ax=ax)
    ax.set_title('Matrice de co-occurrence des Top 15 gestes', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph6_cooccurrence_matrix.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 7: Top 15 transitions (bigrams)
    # ========================================================================
    bigrams = []
    for actions_list in df['actions']:
        for i in range(len(actions_list) - 1):
            bigrams.append(f"{actions_list[i]} → {actions_list[i+1]}")
    
    bigram_counts = Counter(bigrams).most_common(15)
    transitions, counts = zip(*bigram_counts)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = sns.color_palette("rocket_r", 15)
    bars = ax.barh(transitions, counts, color=colors, edgecolor='black', linewidth=1.2)
    ax.set_xlabel('Fréquence', fontsize=12, fontweight='bold')
    ax.set_ylabel('Transition', fontsize=12, fontweight='bold')
    ax.set_title('Top 15 des transitions de gestes les plus fréquentes', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    for i, (bar, count) in enumerate(zip(bars, counts)):
        ax.text(count + max(counts)*0.01, i, f'{count:,}', va='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph7_top15_transitions.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 8: Scatter longueur × diversité
    # ========================================================================
    fig, ax = plt.subplots(figsize=(12, 8))
    scatter = ax.hexbin(df['num_actions'], df['num_unique_actions'], gridsize=30, 
                        cmap='YlOrRd', mincnt=1, edgecolors='black', linewidths=0.2)
    ax.plot([0, df['num_actions'].max()], [0, df['num_actions'].max()], 
            'r--', linewidth=2, label='Diversité maximale (pas de répétition)')
    ax.set_xlabel('Longueur de séquence (nombre total d\'actions)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Nombre de gestes uniques', fontsize=12, fontweight='bold')
    ax.set_title('Diversité du vocabulaire gestuel vs longueur de séquence', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Densité de recettes', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph8_diversite_vocabulaire.png', bbox_inches='tight')
    plt.close()
    
    # ========================================================================
    # GRAPHIQUE 9: Comparaison gestes vs étapes totales
    # ========================================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Scatter plot
    axes[0].scatter(df['number_of_steps'], df['num_actions'], alpha=0.3, c='#4ECDC4', edgecolors='black', linewidths=0.5, s=20)
    axes[0].plot([0, df['number_of_steps'].max()], [0, df['number_of_steps'].max()], 
                 'r--', linewidth=2, label='Égalité (gestes = étapes)')
    axes[0].set_xlabel('Nombre d\'étapes totales (number_of_steps)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Nombre de gestes (len(actions))', fontsize=12, fontweight='bold')
    axes[0].set_title('Comparaison: Gestes extraits vs Étapes totales', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3)
    
    # Histogramme des différences
    df['diff_steps_actions'] = df['number_of_steps'] - df['num_actions']
    axes[1].hist(df['diff_steps_actions'], bins=50, color='#FF6B6B', alpha=0.7, edgecolor='black')
    axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Égalité')
    axes[1].axvline(df['diff_steps_actions'].mean(), color='orange', linestyle='--', linewidth=2, 
                    label=f'Moyenne: {df["diff_steps_actions"].mean():.1f}')
    axes[1].set_xlabel('Différence (étapes - gestes)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Fréquence', fontsize=12, fontweight='bold')
    axes[1].set_title('Distribution des différences', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=10)
    
    plt.suptitle('Évaluation du nombre de gestes extraits par recette', fontsize=15, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/eda_graph9_gestes_vs_steps.png', bbox_inches='tight')
    plt.close()
    
    print(f"✅ {9} graphiques EDA générés avec succès dans {output_dir}/")
    
    # Statistiques résumées
    print("\n" + "="*80)
    print("STATISTIQUES RÉSUMÉES")
    print("="*80)
    print(f"Total recettes: {len(df):,}")
    print(f"Longueur moyenne actions: {df['num_actions'].mean():.2f} ± {df['num_actions'].std():.2f}")
    print(f"Nombre moyen étapes: {df['number_of_steps'].mean():.2f} ± {df['number_of_steps'].std():.2f}")
    print(f"Différence moyenne (étapes - gestes): {df['diff_steps_actions'].mean():.2f}")
    print(f"Gestes uniques totaux: {len(set(all_actions))}")
    print("="*80)

if __name__ == "__main__":
    # Exemple d'utilisation (à adapter avec tes vrais dataframes)
    # df_graphs = pd.read_csv('path_to_graphs.csv')
    # df_metadata = pd.read_csv('path_to_metadata.csv')
    run_eda_pipeline(final_df, recipes_df)
    

✅ 9 graphiques EDA générés avec succès dans C:/Users/nguem/Downloads/recipes-20250605T021334Z-1-002/recipes/EDA_outputs/

STATISTIQUES RÉSUMÉES
Total recettes: 2,308,155
Longueur moyenne actions: 8.59 ± 4.94
Nombre moyen étapes: 10.49 ± 6.68
Différence moyenne (étapes - gestes): 1.91
Gestes uniques totaux: 406
